# 43 — Context Demystifies Forecasting

**Synthesis notebook**

This notebook summarizes the repo's response to arXiv:2606.16076v1, **Phys-JEPA**.

Core claim:

> Forecasting improves when constraints shape hidden state rather than only predictions.

The repo explored that claim through context, toy experiments, decomposition, ablations, climate-style forecasting, interpretability, and long-horizon stability.


## Repo arc

| Notebook | Question | Result |
|---|---|---|
| `00_context` | What problem is Phys-JEPA solving? | Forecasts depend on hidden state. |
| `07_output_vs_latent_constraints` | Do latent constraints outperform output corrections? | Latent constraints reduce drift before decoding. |
| `13_physical_residual_decomposition` | What is being constrained? | Latent state can be decomposed into physical + residual components. |
| `17_climate_forecasting` | Does the idea survive climate-style signals? | Seasonal physical structure improves forecasting. |
| `23_constraint_ablations` | When does latent structure fail? | Constraint quality matters. |
| `29_interpretability` | What does latent state represent? | Physical manifolds make latent state interpretable. |
| `37_forecast_stability` | How does error accumulate? | Latent-state stability produces forecast stability. |


In [ ]:
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

ROOT = Path.cwd()
if ROOT.name == "notebooks":
    ROOT = ROOT.parent

FIGURES = ROOT / "figures"
RESULTS = ROOT / "results"
REPORTS = ROOT / "reports"

FIGURES.mkdir(exist_ok=True)
RESULTS.mkdir(exist_ok=True)
REPORTS.mkdir(exist_ok=True)

print(f"ROOT: {ROOT}")
print(f"FIGURES: {FIGURES}")
print(f"RESULTS: {RESULTS}")
print(f"REPORTS: {REPORTS}")

## Synthesis table

The central comparison is not one model versus another.

The central comparison is **where the constraint acts**.


In [ ]:
synthesis = pd.DataFrame([
    {
        "mode": "unconstrained forecasting",
        "where_constraint_acts": "nowhere",
        "hidden_state": "free to drift",
        "forecast_behavior": "errors compound",
        "interpretability": "low",
        "stability": "low",
    },
    {
        "mode": "output-constrained forecasting",
        "where_constraint_acts": "after prediction",
        "hidden_state": "still unconstrained",
        "forecast_behavior": "bounded but not explanatory",
        "interpretability": "medium-low",
        "stability": "medium",
    },
    {
        "mode": "latent-constrained forecasting",
        "where_constraint_acts": "inside state transition",
        "hidden_state": "structured by measurable variables",
        "forecast_behavior": "drift reduced before decoding",
        "interpretability": "high",
        "stability": "high",
    },
])

synthesis.to_csv(RESULTS / "43_synthesis_table.csv", index=False)
synthesis

In [ ]:
score_map = {
    "low": 1,
    "medium-low": 2,
    "medium": 3,
    "high": 5,
}

plot_df = synthesis.copy()
plot_df["interpretability_score"] = plot_df["interpretability"].map(score_map)
plot_df["stability_score"] = plot_df["stability"].map(score_map)

fig, ax = plt.subplots(figsize=(8, 4.5))

x = np.arange(len(plot_df))
width = 0.35

ax.bar(x - width/2, plot_df["interpretability_score"], width, label="interpretability")
ax.bar(x + width/2, plot_df["stability_score"], width, label="stability")

ax.set_xticks(x)
ax.set_xticklabels(plot_df["mode"], rotation=20, ha="right")
ax.set_ylim(0, 5.5)
ax.set_title("Constraint placement changes interpretability and stability")
ax.set_ylabel("qualitative score")
ax.legend()

fig.tight_layout()
fig.savefig(FIGURES / "43_constraint_placement_scores.png", dpi=160)
plt.show()

## Result snapshots

If prior notebooks have been executed, this section reads their saved result files.

If a file is missing, the notebook reports that and continues.


In [ ]:
def read_csv_if_exists(path):
    path = Path(path)
    if path.exists():
        return pd.read_csv(path)
    print(f"missing: {path.name}")
    return None

metrics_07 = read_csv_if_exists(RESULTS / "07_metrics.csv")
metrics_13 = read_csv_if_exists(RESULTS / "13_decomposition_metrics.csv")
summary_23 = read_csv_if_exists(RESULTS / "23_summary.csv")
metrics_29 = read_csv_if_exists(RESULTS / "29_interpretability_metrics.csv")
metrics_37 = read_csv_if_exists(RESULTS / "37_stability_metrics.csv")

available = {
    "07_metrics": metrics_07 is not None,
    "13_decomposition_metrics": metrics_13 is not None,
    "23_summary": summary_23 is not None,
    "29_interpretability_metrics": metrics_29 is not None,
    "37_stability_metrics": metrics_37 is not None,
}

available

In [ ]:
summary_cards = []

if metrics_07 is not None:
    best = metrics_07.sort_values("RMSE").iloc[0]
    summary_cards.append({
        "notebook": "07",
        "result": "best forecast mode",
        "value": f"{best['method']} (RMSE={best['RMSE']:.3f})",
    })

if metrics_13 is not None:
    best = metrics_13.sort_values("RMSE").iloc[0]
    summary_cards.append({
        "notebook": "13",
        "result": "best decomposition mode",
        "value": f"{best['method']} (RMSE={best['RMSE']:.3f})",
    })

if summary_23 is not None:
    for _, row in summary_23.iterrows():
        summary_cards.append({
            "notebook": "23",
            "result": row["experiment"],
            "value": row["finding"],
        })

if metrics_29 is not None:
    best = metrics_29.sort_values("RMSE").iloc[0]
    summary_cards.append({
        "notebook": "29",
        "result": "best latent trajectory",
        "value": f"{best['method']} (RMSE={best['RMSE']:.3f})",
    })

if metrics_37 is not None:
    best = metrics_37.sort_values("RMSE").iloc[0]
    summary_cards.append({
        "notebook": "37",
        "result": "best stability mode",
        "value": f"{best['method']} (RMSE={best['RMSE']:.3f})",
    })

summary_cards = pd.DataFrame(summary_cards)
summary_cards.to_csv(RESULTS / "43_result_snapshots.csv", index=False)
summary_cards

## Final repo diagram

The repo's logic can be represented as a simple chain:

```text
context
  ↓
latent constraints
  ↓
physical + residual decomposition
  ↓
interpretability
  ↓
forecast stability
```


In [ ]:
steps = [
    "Context",
    "Latent\nconstraints",
    "Physical +\nresidual state",
    "Interpretability",
    "Forecast\nstability",
]

x = np.arange(len(steps))
y = np.zeros(len(steps))

fig, ax = plt.subplots(figsize=(10, 2.8))

ax.scatter(x, y, s=900)

for i, label in enumerate(steps):
    ax.text(x[i], y[i], label, ha="center", va="center", fontsize=10)

for i in range(len(steps) - 1):
    ax.annotate(
        "",
        xy=(x[i+1] - 0.32, y[i+1]),
        xytext=(x[i] + 0.32, y[i]),
        arrowprops=dict(arrowstyle="->", lw=1.8),
    )

ax.set_title("Context Demystifies Forecasting: repo synthesis")
ax.set_axis_off()

fig.tight_layout()
fig.savefig(FIGURES / "43_repo_synthesis_chain.png", dpi=160)
plt.show()

## Engineering lessons

1. **Forecasts depend on hidden state.**  
   A model can make short-term predictions while its latent state drifts.

2. **Output constraints repair after the fact.**  
   They can bound predictions but do not necessarily explain the system.

3. **Latent constraints shape the forecast before decoding.**  
   This reduces drift at the representation level.

4. **Physical/residual decomposition improves interpretability.**  
   The physical component carries measurable structure; the residual component carries remaining variation.

5. **Constraint quality matters.**  
   A weak or wrong constraint can fail. A refined constraint can recover stability.

6. **Forecast stability follows latent-state stability.**  
   Error grows when latent trajectories move away from the physical manifold.


## Lab report statement

> Forecasting improves when constraints shape hidden state rather than only predictions.  
> When measurable structure organizes latent transitions, models become more interpretable, more stable across time horizons, and better aligned with the systems they represent.

```text
latent_state = physical_state + residual_state
forecast = decode(constrained_state_transition)
```


## README recommendation

Use these as the primary README figures:

| Figure | Why |
|---|---|
| `37_horizon_error.png` | strongest stability result |
| `29_constrained_vs_unconstrained_latent.png` | clearest latent-manifold visualization |
| `23_nonstationary_drift.png` | best constraint-quality ablation |
| `13_physical_residual_rollout.png` | best decomposition figure |


In [ ]:
readme_summary = """# forecasting-latent-context

Forecasting improves when constraints shape hidden state rather than only predictions.

Inspired by arXiv:2606.16076v1 — Phys-JEPA: A Physics-informed Joint Embedding Predictive Architecture for Time Series Forecasting.

## Core idea

```text
latent_state = physical_state + residual_state
forecast = decode(constrained_state_transition)
```

## Repo conclusion

Output constraints can repair forecasts after prediction.

Latent constraints shape the representation before prediction.

Across toy systems, climate-style signals, ablations, interpretability views, and long-horizon rollouts, the same pattern appears:

```text
latent-state stability
    ↓
forecast stability
```

## Notebooks

| Notebook | Purpose |
|---|---|
| `00_context.ipynb` | Define the lab report question and toy decomposition. |
| `07_output_vs_latent_constraints.ipynb` | Compare unconstrained, output-constrained, and latent-constrained forecasting. |
| `13_physical_residual_decomposition.ipynb` | Build the anchor physical + residual latent decomposition. |
| `17_climate_forecasting.ipynb` | Move from toy signals to climate-style data. |
| `23_constraint_ablations.ipynb` | Test when latent structure helps, weakens, or fails. |
| `29_interpretability.ipynb` | Visualize physical latent manifolds, residual state, and latent drift. |
| `37_forecast_stability.ipynb` | Study long-horizon error accumulation. |
| `43_context_demystifies_forecasting.ipynb` | Synthesize the repo findings. |

## Lab report

Context Demystifies Forecasting  
https://labreports.app/2606-16076/
"""

(REPORTS / "43_readme_summary.md").write_text(readme_summary)
print(readme_summary[:1000])

## Final conclusion

The repo's answer to Phys-JEPA is:

```text
Put constraints where the model represents the system,
not only where it emits predictions.
```

That sentence connects all notebooks:

```text
context
  → constrained hidden state
  → physical/residual decomposition
  → interpretable latent manifold
  → long-horizon stability
```
